In [1]:
import os
import re
import glob
import numpy as np
import matplotlib.pyplot as plt
import cv2
 
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
 
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [2]:
DATA_DIR = "leapGestRecog"
IMG_SIZE = (64, 64)
RANDOM_STATE = 42
BATCH_SIZE = 32
EPOCHS = 15
 
OUT_DIR = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)
 
GESTURE_NAMES = {
    1: "palm", 2: "l", 3: "fist", 4: "fist_moved", 5: "thumb",
    6: "index", 7: "ok", 8: "palm_moved", 9: "c", 10: "down",
}

In [ ]:
def find_image_paths(data_dir):
    items = []
    gesture_dirs = glob.glob(os.path.join(data_dir, "*", "*"))
    for gdir in gesture_dirs:
        if not os.path.isdir(gdir):
            continue
        folder_name = os.path.basename(gdir)
        m = re.match(r"(\d+)_", folder_name)
        if not m:
            continue
        gesture_num = int(m.group(1))  # 1-10
        label_idx = gesture_num - 1     # 0-9
        label_name = GESTURE_NAMES.get(gesture_num, folder_name)
        for img_path in glob.glob(os.path.join(gdir, "*.png")):
            items.append((img_path, label_idx, label_name))
    return items
 
 
def load_images(items, img_size=IMG_SIZE):
    X, y = [], []
    for path, label_idx, _ in items:
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue
        img = cv2.resize(img, img_size)
        X.append(img)
        y.append(label_idx)
    X = np.array(X, dtype="float32") / 255.0
    X = X.reshape(-1, img_size[0], img_size[1], 1)
    y = np.array(y)
    return X, y
 
 

In [4]:
def build_model(input_shape, num_classes):
    model = keras.Sequential([
        layers.Input(shape=input_shape),
 
        layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
 
        layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
 
        layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
 
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation="softmax"),
    ])
 
    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model
 

In [ ]:
def main():
    print(f"Scanning {DATA_DIR} for images...")
    items = find_image_paths(DATA_DIR)
    print(f"Found {len(items)} images across "
          f"{len(set(l for _, l, _ in items))} gesture classes.")
 
    if len(items) == 0:
        raise FileNotFoundError(
            f"No images found under {DATA_DIR}. "
            "Expected structure: DATA_DIR/<subject>/<NN_gesturename>/*.png. "
            "Update DATA_DIR at the top of the script."
        )
 
    print("Loading and preprocessing images...")
    X, y = load_images(items)
    print(f"Image tensor shape: {X.shape}")
 
    num_classes = len(np.unique(y))
    class_names = [GESTURE_NAMES[i + 1] for i in range(num_classes)]

    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, random_state=RANDOM_STATE, stratify=y_temp
    )
    print(f"Train: {X_train.shape[0]}  Val: {X_val.shape[0]}  Test: {X_test.shape[0]}")

    model = build_model(X_train.shape[1:], num_classes)
    model.summary()
 
    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_accuracy", patience=4, restore_best_weights=True
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=2
        ),
    ]
 
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,
        verbose=2,
    )

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].plot(history.history["accuracy"], label="Train")
    axes[0].plot(history.history["val_accuracy"], label="Validation")
    axes[0].set_title("Accuracy")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()
    axes[0].grid(alpha=0.3)
 
    axes[1].plot(history.history["loss"], label="Train")
    axes[1].plot(history.history["val_loss"], label="Validation")
    axes[1].set_title("Loss")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()
    axes[1].grid(alpha=0.3)
 
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "training_curves.png"), dpi=150)
    plt.close()

    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
    print(f"\n=== Test Set Performance ===")
    print(f"Test Accuracy: {test_acc:.4f}")
    print(f"Test Loss: {test_loss:.4f}")
 
    y_pred_probs = model.predict(X_test, verbose=0)
    y_pred = np.argmax(y_pred_probs, axis=1)
 
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=class_names))

    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    fig, ax = plt.subplots(figsize=(10, 9))
    disp.plot(cmap="Blues", ax=ax, xticks_rotation=45, values_format="d")
    plt.title("Hand Gesture Recognition — Confusion Matrix")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "confusion_matrix.png"), dpi=150)
    plt.close()

    model.save(os.path.join(OUT_DIR, "gesture_cnn_model.keras"))
    print(f"\nSaved model to {OUT_DIR}/gesture_cnn_model.keras")
    print(f"Saved training_curves.png and confusion_matrix.png")
 

In [ ]:
def predict_gesture(image_path, model_path=os.path.join(OUT_DIR, "gesture_cnn_model.keras"),
                     img_size=IMG_SIZE):
    model = keras.models.load_model(model_path)
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, img_size).astype("float32") / 255.0
    img = img.reshape(1, img_size[0], img_size[1], 1)
    pred = model.predict(img, verbose=0)
    class_idx = int(np.argmax(pred))
    return GESTURE_NAMES[class_idx + 1], float(np.max(pred))
 

In [7]:
if __name__ == "__main__":
    main()

Scanning leapGestRecog for images...
Found 20000 images across 10 gesture classes.
Loading and preprocessing images...
Image tensor shape: (20000, 64, 64, 1)
Train: 14000  Val: 3000  Test: 3000


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 64, 64, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 64, 64, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 32, 32, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 32, 32, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 16, 16, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 16, 16, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 8192)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     1,048,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,143,562 (4.36 MB)

 Trainable params: 1,143,114 (4.36 MB)

 Non-trainable params: 448 (1.75 KB)

Epoch 1/15
438/438 - 34s - 78ms/step - accuracy: 0.6800 - loss: 0.8515 - val_accuracy: 0.4643 - val_loss: 2.8688 - learning_rate: 0.0010
Epoch 2/15
438/438 - 28s - 64ms/step - accuracy: 0.9408 - loss: 0.1608 - val_accuracy: 0.9803 - val_loss: 0.0642 - learning_rate: 0.0010
Epoch 3/15
438/438 - 28s - 64ms/step - accuracy: 0.9717 - loss: 0.0828 - val_accuracy: 0.9923 - val_loss: 0.0260 - learning_rate: 0.0010
Epoch 4/15
438/438 - 29s - 66ms/step - accuracy: 0.9805 - loss: 0.0685 - val_accuracy: 0.9990 - val_loss: 0.0039 - learning_rate: 0.0010
Epoch 5/15
438/438 - 29s - 65ms/step - accuracy: 0.9844 - loss: 0.0496 - val_accuracy: 0.9850 - val_loss: 0.0616 - learning_rate: 0.0010
Epoch 6/15
438/438 - 29s - 67ms/step - accuracy: 0.9909 - loss: 0.0283 - val_accuracy: 0.9980 - val_loss: 0.0105 - learning_rate: 0.0010
Epoch 7/15
438/438 - 29s - 66ms/step - accuracy: 0.9938 - loss: 0.0169 - val_accuracy: 1.0000 - val_loss: 2.2659e-04 - learning_rate: 5.0000e-04
Epoch 8/15
438/438 - 38s - 87ms/s